In [1]:
import ee
import os
import geemap
import pandas as p
import geopandas as gpd
from pathlib import Path
import sys
import importlib
import json
import matplotlib.pyplot as plt
import numpy as np


In [2]:
PROJECT_ROOT = Path().resolve().parent

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

True

In [4]:
ee.Authenticate()



Successfully saved authorization token.


In [5]:
ee.Initialize(project=os.environ["EE_PROJECT"])

In [43]:

sys.path.append(str(PROJECT_ROOT / "src"))

from gee_utils import create_images_for_all_locations, get_samples, export_patches


In [24]:
# Want to upload the sites.json to ee and then create image collection for each, clip to the bbox, need to select the dates for each location
# Could save the location in the json or just use the date from the points? 

# Then want to upload the point shapefiles and use those to sample the images for the matching date. Hmmm, should think about that. does each point need a date attribute - yes?
site_fp = PROJECT_ROOT / "configs" / "sites.json"
points_fp = PROJECT_ROOT / "configs" / "point_files" / "mula_points.shp"


In [44]:
image_collection = create_images_for_all_locations(sites_file= site_fp, project_root= PROJECT_ROOT)

Created image collection for: Vembanad
Created image collection for: Winam
Created image collection for: Inle
Created image collection for: Hartbeespoort
Created image collection for: Mula
Created image collection for: RawaPening
Created image collection for: Rodman
Created image collection for: Valsequillo


In [45]:
samples = get_samples(merged_ic=image_collection,sites_file= site_fp, project_root=PROJECT_ROOT)

In [46]:
samples.tail()

,B11,B12,B2,B3,B4,B5,B6,B7,B8,B8A,binary,lat,lc,location,lon,obs_date
1712,2710,2587,1424,1870,2134,2096,2180,2276,2676,2497,1,18.947760,1,Valsequillo,-98.225521,2020-12-27
1713,1610,980,242,448,462,923,1504,1802,1942,2173,1,18.940574,1,Valsequillo,-98.206477,2020-12-27
1714,1653,794,264,623,448,1030,3159,3767,4006,3999,1,18.939406,1,Valsequillo,-98.236660,2020-12-27
1715,1869,1396,513,649,827,949,1144,1317,1359,1513,1,18.934914,1,Valsequillo,-98.196416,2020-12-27
1716,1511,961,224,355,442,661,870,976,1025,1154,1,18.930782,1,Valsequillo,-98.226599,2020-12-27


In [49]:
len(samples.loc[samples["binary"] == 1.0])

461

In [50]:
samples_outpath = PROJECT_ROOT / "outputs" / "sample_points.csv"

samples.to_csv(samples_outpath)

In [51]:
samples.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116 entries, 0 to 115
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   B11       116 non-null    int64  
 1   B12       116 non-null    int64  
 2   B2        116 non-null    int64  
 3   B3        116 non-null    int64  
 4   B4        116 non-null    int64  
 5   B5        116 non-null    int64  
 6   B6        116 non-null    int64  
 7   B7        116 non-null    int64  
 8   B8        116 non-null    int64  
 9   B8A       116 non-null    int64  
 10  lat       116 non-null    float64
 11  lc        114 non-null    float64
 12  location  116 non-null    object 
 13  lon       116 non-null    float64
 14  obs_date  116 non-null    object 
dtypes: float64(3), int64(10), object(2)
memory usage: 13.7+ KB


In [ ]:
mula_wh_samples = samples.loc[(samples["location"]=="mula") & (samples["lc"]== 1.0), ["B11", "B12", "B3", "B4", "B5", "B8"]]
mula_non_wh_samples = samples.loc[(samples["location"]=="mula")& (samples["lc"]==0.0), ["B11", "B12", "B3", "B4", "B5", "B8"]]
bins = np.linspace(0,8000, 20)

for col in ["B11", "B12", "B3", "B4", "B5", "B8"]:
    plt.hist(mula_wh_samples[col], bins, alpha=0.5, label='x')
    plt.hist(mula_non_wh_samples[col], bins, alpha=0.5, label='y')
    plt.legend(loc='upper right')
    plt.title(f"{col}")
    plt.show()



In [15]:
export_patches(merged_ic = image_collection, sites_file= site_fp, project_root=PROJECT_ROOT)

Exporting sampled patches to drive/Dissertation


In [42]:
import importlib
import gee_utils

importlib.reload(gee_utils)


<module 'gee_utils' from '/Users/bensutton/Library/CloudStorage/Dropbox/MASTERS/Dissertation/code/src/gee_utils.py'>